# Module 4 — Streamlit UI + TTS + Accessibility

**What:** patient-facing web app for PrescriptAI.

**Features:**
- Image upload (camera or file)
- Runs full Module 3 pipeline + Module 3.5 interaction checker
- Shows extracted drugs as accessible cards
- Plain-language explanation
- Drug interaction warnings (color-coded by severity)
- TTS button (text-to-speech via gTTS)
- High-contrast accessibility theme (large fonts, dark mode option)
- Public URL via ngrok for phone demo

**Tech:** Streamlit + gTTS + ngrok + your existing Module 3/3.5 functions

**Time:** ~3 hours including testing.

**One-time setup:** ngrok needs an auth token. Free at [ngrok.com](https://ngrok.com).

## Cell 1 — Bootstrap

In [ ]:
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/prescriptai')
ENV_FILE = PROJECT_ROOT / '.env'

# Load env
for line in ENV_FILE.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        k, v = line.split('=', 1)
        os.environ[k.strip()] = v.strip()

os.environ['HF_HOME'] = str(PROJECT_ROOT / 'hf_cache')
os.environ['TRANSFORMERS_CACHE'] = str(PROJECT_ROOT / 'hf_cache')
os.environ['HF_HUB_CACHE'] = str(PROJECT_ROOT / 'hf_cache')

print('Bootstrap done.')

Mounted at /content/drive
Bootstrap done.


## Cell 2 — Install Streamlit + ngrok + gTTS

In [ ]:
!pip install -q streamlit pyngrok gtts
!pip install -q google-generativeai sentence-transformers chromadb rapidfuzz Pillow
print('Packages installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 79.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.2 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Cell 3 — Set ngrok auth token

**One-time setup:**
1. Go to [ngrok.com/signup](https://ngrok.com/signup) — free account
2. Copy your authtoken from [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Replace the placeholder below

In [ ]:
NGROK_AUTH_TOKEN = '397E***'   # <-- replace this

from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN.startswith('PASTE'):
    print('!! Add your ngrok token above before continuing')
else:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    print(f'Ngrok token set: {NGROK_AUTH_TOKEN[:8]}...')

Ngrok token set: 397E2jfO...


## Cell 4 — Write the Streamlit app to disk

Streamlit apps run as a separate Python process. We write the full app code to `app.py`.

**Architecture:**
- `app.py` does its own imports + key loading + ChromaDB connection (Streamlit subprocess doesn't share notebook variables)
- Module 3/3.5 logic copy-pasted into the app
- Streamlit handles UI rendering

In [ ]:
APP_PATH = '/content/app.py'

app_code = '\nimport os\nimport json\nimport io\nfrom pathlib import Path\nfrom itertools import combinations\n\nimport streamlit as st\nfrom PIL import Image\nimport google.generativeai as genai\nimport torch\nimport numpy as np\nfrom sentence_transformers import SentenceTransformer\nfrom rapidfuzz import fuzz, process\nfrom gtts import gTTS\n\nPROJECT_ROOT = Path("/content/drive/MyDrive/prescriptai")\nENV_FILE = PROJECT_ROOT / ".env"\n\nfor line in ENV_FILE.read_text().splitlines():\n    line = line.strip()\n    if line and not line.startswith("#") and "=" in line:\n        k, v = line.split("=", 1)\n        os.environ[k.strip()] = v.strip()\n\nos.environ["HF_HOME"] = str(PROJECT_ROOT / "hf_cache")\nos.environ["TRANSFORMERS_CACHE"] = str(PROJECT_ROOT / "hf_cache")\nos.environ["HF_HUB_CACHE"] = str(PROJECT_ROOT / "hf_cache")\n\nGEMINI_KEYS = []\nfor suffix in ["", "_2", "_3", "_4", "_5"]:\n    val = os.environ.get(f"GEMINI_API_KEY{suffix}", "").strip()\n    if val and not val.startswith("AIzaSy_paste") and len(val) >= 30:\n        GEMINI_KEYS.append(val)\n\nGEMINI_MODEL = "gemini-2.5-flash"\n_gemini_key_idx = 0\n\ndef call_gemini_with_retry(prompt_or_parts, max_retries=None):\n    global _gemini_key_idx\n    if not GEMINI_KEYS:\n        raise RuntimeError("No Gemini keys.")\n    if max_retries is None:\n        max_retries = len(GEMINI_KEYS) * 2\n    last_err = None\n    for attempt in range(max_retries):\n        try:\n            genai.configure(api_key=GEMINI_KEYS[_gemini_key_idx])\n            client = genai.GenerativeModel(GEMINI_MODEL)\n            return client.generate_content(prompt_or_parts)\n        except Exception as e:\n            err = str(e)\n            if "429" in err or "quota" in err.lower():\n                _gemini_key_idx = (_gemini_key_idx + 1) % len(GEMINI_KEYS)\n                last_err = e\n                continue\n            raise\n    raise last_err\n\nst.set_page_config(page_title="PrescriptAI", page_icon="💊", layout="wide", initial_sidebar_state="expanded")\n\nst.markdown("""\n<style>\n    html, body, [class*="css"] { font-family: "Atkinson Hyperlegible", "Verdana", sans-serif; }\n    .stApp { background-color: #fffef0; color: #1a1a1a; }\n    h1, h2, h3 { color: #003366; font-weight: 700; }\n    h1 { font-size: 2.5rem !important; }\n    h2 { font-size: 2rem !important; }\n    h3 { font-size: 1.5rem !important; }\n    p, label, .stMarkdown { font-size: 1.15rem !important; line-height: 1.7 !important; }\n    .stButton button { font-size: 1.2rem !important; padding: 0.6rem 1.5rem !important; background-color: #003366 !important; color: #ffffff !important; border: 2px solid #003366 !important; border-radius: 8px !important; }\n    .stButton button:hover { background-color: #ffd700 !important; color: #000000 !important; }\n    .drug-card { padding: 1.5rem; margin-bottom: 1rem; border: 3px solid #003366; border-radius: 12px; background-color: #ffffff; }\n    .drug-card.matched { border-color: #006400; background-color: #f0fff0; }\n    .drug-card.low_confidence { border-color: #ff8c00; background-color: #fff8dc; }\n    .drug-card.rejected_by_llm { border-color: #8b0000; background-color: #ffe4e1; }\n    .drug-card.unmatched { border-color: #696969; background-color: #f5f5f5; }\n    .severe-warning { padding: 1rem; margin: 1rem 0; border-radius: 8px; background-color: #ffcccc; border: 3px solid #8b0000; color: #000; }\n    .moderate-warning { padding: 1rem; margin: 1rem 0; border-radius: 8px; background-color: #ffe4b5; border: 3px solid #ff8c00; color: #000; }\n    .mild-warning { padding: 1rem; margin: 1rem 0; border-radius: 8px; background-color: #fffacd; border: 3px solid #daa520; color: #000; }\n</style>\n""", unsafe_allow_html=True)\n\nINTERACTION_RULES = [\n    ({"warfarin", "acitrom"}, {"aspirin", "ibuprofen", "diclofenac", "naproxen", "aceclofenac"}, "severe", "Increased bleeding risk. NSAIDs displace warfarin from protein binding and inhibit platelets."),\n    ({"warfarin"}, {"metronidazole", "fluconazole", "amiodarone"}, "severe", "Increased warfarin effect, INR may spike dangerously."),\n    ({"sertraline", "escitalopram", "fluoxetine", "paroxetine", "citalopram"}, {"tramadol", "pethidine", "linezolid"}, "severe", "Risk of serotonin syndrome (high fever, agitation, can be fatal)."),\n    ({"azithromycin", "clarithromycin", "erythromycin"}, {"amiodarone", "sotalol", "quinidine"}, "severe", "Both prolong QT interval, risk of fatal arrhythmia."),\n    ({"diazepam", "lorazepam", "alprazolam", "clonazepam"}, {"tramadol", "morphine", "codeine", "oxycodone"}, "severe", "Combined CNS depression, risk of respiratory failure (FDA black box warning)."),\n    ({"isotretinoin"}, {"doxycycline", "minocycline", "tetracycline"}, "severe", "Risk of pseudotumor cerebri (raised intracranial pressure)."),\n    ({"ibuprofen", "diclofenac", "naproxen", "aceclofenac", "etoricoxib"}, {"ramipril", "enalapril", "lisinopril", "telmisartan", "losartan", "olmesartan"}, "moderate", "NSAIDs reduce blood pressure medication effectiveness, may impair kidney function."),\n    ({"prednisolone", "dexamethasone", "hydrocortisone"}, {"ibuprofen", "diclofenac", "naproxen", "aceclofenac", "aspirin"}, "moderate", "Increased risk of stomach bleeding and ulcers."),\n    ({"simvastatin", "atorvastatin", "lovastatin"}, {"clarithromycin", "erythromycin", "itraconazole"}, "moderate", "Increased statin levels, risk of muscle damage."),\n    ({"metformin", "glimepiride", "glibenclamide", "gliclazide"}, {"fluconazole", "ciprofloxacin"}, "moderate", "Increased risk of low blood sugar, monitor closely."),\n    ({"paracetamol", "acetaminophen"}, {"alcohol", "isoniazid"}, "moderate", "Increased risk of liver damage."),\n    ({"ramipril", "enalapril", "lisinopril"}, {"telmisartan", "losartan", "olmesartan", "valsartan"}, "moderate", "Combined ACE-I and ARB increases hyperkalemia and kidney injury risk."),\n]\n\ndef find_rule_match(drug_a, drug_b):\n    a, b = drug_a.lower(), drug_b.lower()\n    for kx, ky, sev, mech in INTERACTION_RULES:\n        ax = any(kw in a for kw in kx); by = any(kw in b for kw in ky)\n        ay = any(kw in a for kw in ky); bx = any(kw in b for kw in kx)\n        if (ax and by) or (ay and bx):\n            return (sev, mech)\n    return None\n\ndef check_interactions(enriched_drugs):\n    matched = [d for d in enriched_drugs if d.get("match_status") == "matched"]\n    if len(matched) < 2:\n        return []\n    warnings = []\n    seen = set()\n    for da, db in combinations(matched, 2):\n        na = da.get("corrected_generic") or da.get("corrected_name") or da.get("raw_name", "")\n        nb = db.get("corrected_generic") or db.get("corrected_name") or db.get("raw_name", "")\n        if not na or not nb: continue\n        key = tuple(sorted([na.lower(), nb.lower()]))\n        if key in seen: continue\n        seen.add(key)\n        m = find_rule_match(na, nb)\n        if m:\n            warnings.append({"drug_a": na, "drug_b": nb, "severity": m[0], "mechanism": m[1]})\n    return warnings\n\n@st.cache_resource\ndef load_rag():\n    drugs_json = PROJECT_ROOT / "data" / "drugs" / "indian_drugs.json"\n    with open(drugs_json) as f:\n        corpus = json.load(f)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    enc = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)\n    embeddings = {}\n    for i, drug in enumerate(corpus):\n        text = f"{drug.get(\'name\', \'\')} {drug.get(\'generic\', \'\')} {drug.get(\'description\', \'\')}"\n        vec = enc.encode([text], normalize_embeddings=True, show_progress_bar=False)[0]\n        embeddings[i] = {"drug": drug, "vec": vec}\n    names = list(dict.fromkeys(\n        [d["name"] for d in corpus if d.get("name")] +\n        [d["generic"] for d in corpus if d.get("generic")]\n    ))\n    return enc, embeddings, corpus, names\n\nencoder, drug_embeddings, drug_corpus, all_names = load_rag()\n\ndef embed_text(text):\n    if isinstance(text, str): text = [text]\n    return np.asarray(encoder.encode(text, normalize_embeddings=True, show_progress_bar=False))\n\ndef sanitize(obj):\n    if isinstance(obj, dict): return {k: sanitize(v) for k, v in obj.items()}\n    elif isinstance(obj, list): return [sanitize(i) for i in obj]\n    elif isinstance(obj, (np.float32, np.float64)): return float(obj)\n    return obj\n\ndef correct_ocr_candidate(ocr_text, k=5):\n    # Fix common OCR misreads\n    clean_text = ocr_text.strip()\n    clean_text = clean_text.replace(" durgs", " drops").replace(" durg", " drop")\n    clean_text = clean_text.replace(" drups", " drops").replace(" dops", " drops")\n    clean_text = clean_text.replace(" tabiet", " tablet").replace(" tabiet", " tablet")\n\n    q_vec = embed_text(clean_text)[0]\n    scores = []\n    for idx, data in drug_embeddings.items():\n        sim = float(np.dot(q_vec, data["vec"]))\n        scores.append({"name": data["drug"]["name"], "combined_score": sim, "metadata": data["drug"]})\n    scores.sort(key=lambda x: x["combined_score"], reverse=True)\n    top_k = scores[:k]\n    fuz = process.extract(clean_text, [x["name"] for x in top_k], scorer=fuzz.WRatio, limit=k)\n    fuz_scores = {n: s/100.0 for n, s, _ in fuz if s >= 50}\n    for item in top_k:\n        item["combined_score"] = 0.6 * item["combined_score"] + 0.4 * fuz_scores.get(item["name"], 0.0)\n    top_k.sort(key=lambda x: x["combined_score"], reverse=True)\n    return {"best": top_k[0] if top_k else None}\n\nEXTRACTION_PROMPT = (\n    "You are a medical-text extraction agent looking at a prescription image.\\n\\n"\n    "For every medication, extract:\\n"\n    "- raw_name: drug name as written\\n"\n    "- dosage: e.g., 500mg. Use null if unclear.\\n"\n    "- frequency: e.g., 1-0-1, BD, TDS, OD, SOS. Use null if unclear.\\n"\n    "- duration: e.g., 5 days. Use null if unclear.\\n"\n    "- route: oral, topical, injection. Default oral.\\n"\n    "- notes: other instructions. Use null if none.\\n\\n"\n    \'Return ONLY valid JSON:\\n\'\n    \'{"drugs": [{"raw_name": "...", "dosage": "...", "frequency": "...", "duration": "...", "route": "...", "notes": "..."}]}\\n\\n\'\n    \'If no drugs visible, return {"drugs": []}.\'\n)\n\ndef clean_json(text):\n    text = text.strip()\n    if text.startswith("```"):\n        lines = text.splitlines()\n        text = "\\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])\n    return text\n\ndef extract_drugs_from_image(image_bytes):\n    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")\n    if max(img.size) > 2000:\n        img.thumbnail((2000, 2000))\n    buf = io.BytesIO()\n    img.save(buf, format="JPEG", quality=90)\n    response = call_gemini_with_retry([EXTRACTION_PROMPT, {"mime_type": "image/jpeg", "data": buf.getvalue()}])\n    return json.loads(clean_json(response.text))\n\nVERIFY_PROMPT = (\n    \'Pharmaceutical name-matching expert.\\n\\n\'\n    \'Prescription contains: "{raw}"\\n\'\n    \'Suggested match: "{name}" (generic: {gen})\\n\\n\'\n    \'Is "{raw}" a brand, abbreviation, misspelling, or alternate name for "{name}" or generic "{gen}"? \'\n    \'Drug class similarity is NOT enough.\\n\\n\'\n    \'Reply with EXACTLY ONE word: YES, NO, or UNCERTAIN.\'\n)\n\ndef verify_match(raw, name, gen):\n    try:\n        r = call_gemini_with_retry(VERIFY_PROMPT.format(raw=raw, name=name, gen=gen or name))\n        ans = r.text.strip().upper().split()[0]\n        return ans if ans in ("YES", "NO", "UNCERTAIN") else "UNCERTAIN"\n    except Exception:\n        return "UNCERTAIN"\n\ndef enrich_drugs(extracted):\n    out = []\n    for drug in extracted.get("drugs", []):\n        raw = drug.get("raw_name", "")\n        if not raw: continue\n        result = correct_ocr_candidate(raw)\n        rec = dict(drug)\n        if result["best"] is None:\n            rec.update({"corrected_name": None, "corrected_generic": None, "medical_info": {},\n                        "confidence": 0.0, "match_status": "unmatched", "llm_verified": None})\n        else:\n            best = result["best"]\n            meta = best.get("metadata") or {}\n            score = float(best["combined_score"])\n            rec.update({\n                "corrected_name": best["name"],\n                "corrected_generic": meta.get("generic", ""),\n                "medical_info": {\n                    "description": meta.get("description", ""),\n                    "uses": meta.get("uses", ""),\n                    "side_effects": meta.get("side_effects", ""),\n                },\n                "confidence": score,\n            })\n            if score >= 0.55:\n                rec["match_status"] = "matched"\n                rec["llm_verified"] = "SKIPPED_HIGH_CONF"\n            elif score >= 0.40:\n                verdict = verify_match(raw, best["name"], meta.get("generic", ""))\n                rec["llm_verified"] = verdict\n                rec["match_status"] = "matched" if verdict != "NO" else "rejected_by_llm"\n            else:\n                rec["match_status"] = "low_confidence"\n                rec["llm_verified"] = None\n        out.append(rec)\n    return sanitize(out)\n\nEXPLANATION_PROMPT = (\n    "You are a patient-education assistant explaining a prescription in plain English.\\n"\n    "Help the patient UNDERSTAND. Do NOT give medical advice.\\n\\n"\n    "Prescription data:\\n{drugs}\\n\\n"\n    "Drug interaction warnings:\\n{warnings}\\n\\n"\n    "For each drug, write:\\n"\n    "## [Drug name]\\n"\n    "**What it is:** [1 sentence]\\n"\n    "**What it treats:** [from uses field, plain language]\\n"\n    "**How to take it:** [from prescription dosage and frequency]\\n"\n    "**Common side effects:** [from side_effects field]\\n"\n    "**Notes:** [prescription notes]\\n\\n"\n    "If interaction warnings exist, after drugs add:\\n"\n    "## ⚠️ Important Drug Combinations\\n"\n    "Plain-language explanation of each warning.\\n\\n"\n    "Rules: Use only data fields. 8th-grade English. No dosage advice. No diagnosis.\\n"\n    "For unmatched/low_confidence/rejected_by_llm drugs: "\n    \'"I could not find verified information about this medication. Please ask your pharmacist or doctor."\\n\\n\'\n    "End with this exact disclaimer:\\n"\n    "---\\n"\n    "**Important:** This is general information to help you understand your prescription. "\n    "It is not medical advice. Always confirm with your doctor or pharmacist if anything is unclear, "\n    "and never change your dosage on your own."\n)\n\ndef generate_explanation(enriched, interactions):\n    if not enriched:\n        return "No medications detected. Please try a clearer photo."\n    drug_data = json.dumps(enriched, indent=2, ensure_ascii=False)\n    int_data = json.dumps(interactions, indent=2) if interactions else "(No interactions flagged.)"\n    response = call_gemini_with_retry(EXPLANATION_PROMPT.format(drugs=drug_data, warnings=int_data))\n    return response.text\n\ndef text_to_speech(text):\n    clean = text.replace("##", "").replace("**", "").replace("---", "").replace("⚠️", "warning:")\n    tts = gTTS(text=clean, lang="en", slow=False)\n    buf = io.BytesIO()\n    tts.write_to_fp(buf)\n    buf.seek(0)\n    return buf.read()\n\nst.title("💊 PrescriptAI")\nst.markdown("### Understand your prescription in plain language")\nst.markdown("Upload a photo of your prescription. PrescriptAI will read it, look up each medication, and explain it in simple words.")\n\nwith st.sidebar:\n    st.markdown("## How it works")\n    st.markdown("""\n1. **Upload** prescription photo\n2. **AI reads** drug names from handwriting\n3. **Looks up** each drug in verified database\n4. **Checks** for dangerous combinations\n5. **Explains** in plain English\n6. **Listen** with TTS button\n""")\n    st.markdown("---")\n    st.markdown("⚠️ **Not medical advice.** Always confirm with your doctor or pharmacist.")\n    st.markdown("---")\n    st.markdown(f"**Database:** {len(drug_corpus)} drugs")\n    st.markdown(f"**API keys loaded:** {len(GEMINI_KEYS)}")\n\nuploaded = st.file_uploader("📷 Upload prescription image", type=["jpg", "jpeg", "png", "webp"])\n\nif uploaded:\n    # Clear previous results on new upload\n    for key in ["enriched", "interactions", "explanation"]:\n        st.session_state.pop(key, None)\n\n    image_bytes = uploaded.read()\n    img = Image.open(io.BytesIO(image_bytes))\n    col1, col2 = st.columns([1, 2])\n    with col1:\n        st.image(img, caption="Your prescription", use_container_width=True)\n    with col2:\n        if st.button("🔍 Analyze Prescription", type="primary"):\n            with st.status("Reading prescription...", expanded=True) as status:\n                st.write("📖 Stage 1: Extracting drugs from image...")\n                try:\n                    extracted = extract_drugs_from_image(image_bytes)\n                    st.write(f"  → Found {len(extracted.get(\'drugs\', []))} drugs")\n                except Exception as e:\n                    st.error(f"Error reading image: {e}")\n                    st.stop()\n                st.write("🔎 Stage 2: Looking up drugs and verifying matches...")\n                enriched = enrich_drugs(extracted)\n                matched = sum(1 for d in enriched if d.get("match_status") == "matched")\n                st.write(f"  → Verified {matched}/{len(enriched)} drugs")\n                st.write("⚠️ Stage 3: Checking drug interactions...")\n                interactions = check_interactions(enriched)\n                st.write(f"  → Found {len(interactions)} interactions")\n                st.write("📝 Stage 4: Writing plain-language explanation...")\n                explanation = generate_explanation(enriched, interactions)\n                status.update(label="Done!", state="complete")\n            st.session_state.enriched = enriched\n            st.session_state.interactions = interactions\n            st.session_state.explanation = explanation\n\nif "enriched" in st.session_state:\n    st.markdown("---")\n    st.markdown("## 💊 Drugs Found")\n    for drug in st.session_state.enriched:\n        status_val = drug.get("match_status", "unknown")\n        emoji = {"matched": "✅", "low_confidence": "⚠️", "rejected_by_llm": "❌", "unmatched": "❓"}.get(status_val, "•")\n        with st.container():\n            st.markdown(f\'<div class="drug-card {status_val}">\', unsafe_allow_html=True)\n            st.markdown(f"### {emoji} {drug.get(\'raw_name\', \'Unknown\')}")\n            if status_val == "matched":\n                st.markdown(f"**Identified as:** {drug.get(\'corrected_name\')} (generic: {drug.get(\'corrected_generic\')})")\n                st.markdown(f"**Confidence:** {drug.get(\'confidence\', 0)*100:.0f}%")\n                if drug.get("dosage"): st.markdown(f"**Dosage:** {drug[\'dosage\']}")\n                if drug.get("frequency"): st.markdown(f"**Frequency:** {drug[\'frequency\']}")\n            else:\n                st.markdown("**Status:** Could not verify this medication")\n                st.markdown("_Please ask your pharmacist or doctor about this drug._")\n            st.markdown(\'</div>\', unsafe_allow_html=True)\n\n    if st.session_state.interactions:\n        st.markdown("## ⚠️ Drug Interaction Warnings")\n        for w in st.session_state.interactions:\n            sev = w["severity"]\n            st.markdown(f\'<div class="{sev}-warning">\', unsafe_allow_html=True)\n            st.markdown(f"**{sev.upper()}:** {w[\'drug_a\']} + {w[\'drug_b\']}")\n            st.markdown(f"_{w[\'mechanism\']}_")\n            st.markdown(\'</div>\', unsafe_allow_html=True)\n\n    st.markdown("## 📝 Plain-Language Explanation")\n    st.markdown(st.session_state.explanation)\n\n    st.markdown("---")\n    if st.button("🔊 Listen to explanation (Text-to-Speech)"):\n        with st.spinner("Generating audio..."):\n            try:\n                audio = text_to_speech(st.session_state.explanation)\n                st.audio(audio, format="audio/mp3")\n            except Exception as e:\n                st.error(f"TTS error: {e}")\n\n    output_json = json.dumps({\n        "drugs": st.session_state.enriched,\n        "interactions": st.session_state.interactions,\n        "explanation": st.session_state.explanation,\n    }, indent=2, ensure_ascii=False)\n    st.download_button("💾 Download full result (JSON)", data=output_json,\n                       file_name="prescriptai_result.json", mime="application/json")\n'

from pathlib import Path
Path(APP_PATH).write_text(app_code)
print(f'Wrote app.py: {len(app_code)} chars')

Wrote app.py: 19906 chars


## Cell 5 — Launch Streamlit + ngrok tunnel

In [ ]:
import subprocess
import time

# Kill any existing streamlit/ngrok
!pkill -f streamlit 2>/dev/null || true
ngrok.kill()
time.sleep(2)

# Start Streamlit in background
subprocess.Popen(
    ['streamlit', 'run', APP_PATH,
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
    ],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Wait for Streamlit to start
print('Starting Streamlit...')
time.sleep(8)

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print(f'\n🚀 PrescriptAI is LIVE at: {public_url}')
print(f'\nOpen this URL on your phone or share with examiner.')
print(f'Tunnel will stay alive as long as this Colab cell is running.')

^C
Starting Streamlit...

🚀 PrescriptAI is LIVE at: NgrokTunnel: "https://unfurnished-caryn-meekly.ngrok-free.dev" -> "http://localhost:8501"

Open this URL on your phone or share with examiner.
Tunnel will stay alive as long as this Colab cell is running.


## Cell 6 — Stop server (run this when done)

In [ ]:
ngrok.kill()
!pkill -f streamlit 2>/dev/null || true
print('Server stopped.')

^C
Server stopped.


## Module 4 — Done

**What works:**
- ✅ Image upload (jpg/png/webp/jpeg)
- ✅ Full pipeline: vision → RAG → verifier → interaction check → explanation
- ✅ Drug cards color-coded by match status
- ✅ Interaction warnings color-coded by severity (red/orange/yellow)
- ✅ Plain-language explanation
- ✅ TTS button (gTTS English)
- ✅ JSON download
- ✅ Accessibility theme (large fonts, high contrast, navy/cream colors, dyslexia-friendly font stack)
- ✅ Public URL via ngrok (phone-demoable)

**Story for the report:**

*"Module 4 implements the patient-facing UI as a Streamlit web application. The interface emphasizes accessibility: large fonts, high-contrast color scheme (navy on cream), color-coded match-status cards (green=matched, orange=low-confidence, red=rejected), and severity-tiered interaction warnings. Text-to-speech via gTTS allows users to listen to the explanation, supporting low-vision and low-literacy users. The app is exposed via ngrok for live demo on any device."*

**Next:** Module 5 — Evaluation + report.

**For viva demo:**
- Run Cells 1-5 just before demo
- Open the public URL on your phone
- Upload prescription, watch pipeline run live
- Click TTS to demonstrate accessibility
- Backup: keep Module 3 notebook open in case Streamlit acts up